In [1]:
# Dataset report and ablation for the two block run.
# This notebook is used for dataset reporting and the DC tail ablation comparison.

import os
import re
import glob
import pickle
import json
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Paths used throughout the report.
OUT_DIR = r"D:\EEG_CleanSegments_And_Datasets_AUX"

# Files written by the main preprocessing notebook that are useful for traceability.
RECORDINGS_INDEX_PATH = os.path.join(OUT_DIR, "recordings_index.csv")
BUILD_SUMMARY_PATH = os.path.join(OUT_DIR, "dataset_build_summary.json")

# Load the recommended DC tail value if it has already been estimated.
DC_TAIL_CONFIG_PATH = os.path.join(OUT_DIR, "DC_TAIL_S_recommended.json")
DC_TAIL_S = None
if os.path.exists(DC_TAIL_CONFIG_PATH):
    try:
        with open(DC_TAIL_CONFIG_PATH, "r", encoding="utf-8") as f:
            _cfg = json.load(f)
        if _cfg.get("DC_TAIL_S", None) is not None:
            DC_TAIL_S = float(_cfg["DC_TAIL_S"])
            print(f"[DC_TAIL] Found DC_TAIL_S={DC_TAIL_S:.3f}s in: {DC_TAIL_CONFIG_PATH}")
        else:
            print(f"[DC_TAIL] Config found but key DC_TAIL_S missing: {DC_TAIL_CONFIG_PATH}")
    except Exception as e:
        print("[DC_TAIL] Could not read config:", e)
else:
    print(f"[DC_TAIL] Config not found at: {DC_TAIL_CONFIG_PATH} (report will proceed without it)")

# Per recording metadata exported by the preprocessing pipeline.
META_GLOB = os.path.join(OUT_DIR, "*_segments_meta.pkl")

# Optional datasets for the DC tail ablation.
NPZ_NO_DCTAIL   = os.path.join(OUT_DIR, "clean_time_dataset_no_dctail.npz")
NPZ_WITH_DCTAIL = os.path.join(OUT_DIR, "clean_time_dataset_with_dctail.npz")

# Output folders for the report tables and figures.
REPORT_DIR = os.path.join(OUT_DIR, "_dataset_report")
PLOT_DIR = os.path.join(REPORT_DIR, "plots")
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

print("OUT_DIR:", OUT_DIR)
print("Meta files found (raw glob):", len(glob.glob(META_GLOB)))
print("Recordings index exists:", os.path.exists(RECORDINGS_INDEX_PATH))
print("Build summary exists:", os.path.exists(BUILD_SUMMARY_PATH))

[DC_TAIL] Found DC_TAIL_S=1.836s in: D:\EEG_CleanSegments_And_Datasets_AUX\DC_TAIL_S_recommended.json
OUT_DIR: D:\EEG_CleanSegments_And_Datasets_AUX
Meta files found (raw glob): 59
Recordings index exists: True
Build summary exists: True


In [2]:
# This block repeats the same stimulus parsing rules used in the main pipeline.
# So the reporting notebook stays fully consistent with dataset construction.

# Three class coding used in the main preprocessing pipeline.
CODES_MAP = {
    "S11": 0,     # Left real stimulus.
    "S110": 0,    # Left sham stimulus.
    "S12": 1,     # Right real stimulus.
    "S120": 1,    # Right sham stimulus.
    "S13": 2,     # No stimulus.
}

STIM_CODE_RE = re.compile(r"^Stimulus/\s*(S)\s*([0-9]+)\s*$", re.IGNORECASE)
RECORDING_ID_RE = re.compile(r"(.+?)_block(\d+)$", re.IGNORECASE)

def parse_stim_code(desc: str):
    m = STIM_CODE_RE.match((desc or "").strip())
    if not m:
        return None
    return f"{m.group(1).upper()}{m.group(2)}"

def is_task_stimulus_desc(desc: str) -> bool:
    code = parse_stim_code(desc)
    return (code is not None) and (code in CODES_MAP)

def _safe_int_or_none(x):
    try:
        if x is None:
            return None
        if isinstance(x, str) and x.strip() == "":
            return None
        return int(x)
    except Exception:
        return None

def parse_recording_id(recording_id: str):
    """
    Example recording_id: 02gd_block1

    Returns:
        participant, block, recording_id
    """
    text = os.path.splitext(os.path.basename(str(recording_id)))[0]
    m = RECORDING_ID_RE.match(text)
    if m:
        return m.group(1), int(m.group(2)), text
    return text, None, text

def infer_identifiers_from_meta(meta_row: dict, fallback_name: str = "unknown"):
    """
    Recover participant, block, recording_id, and file_path from the metadata row.

    This is written to handle the newer metadata format, where subject, block_id,
    recording_id, and file_path may appear as separate fields.
    """
    meta_row = {} if meta_row is None else dict(meta_row)

    subject = str(meta_row.get("subject", "") or "").strip()
    recording_id = str(meta_row.get("recording_id", "") or "").strip()
    file_path = str(meta_row.get("file_path", "") or "").strip()
    block = _safe_int_or_none(meta_row.get("block_id", None))

    if recording_id == "":
        if file_path != "":
            recording_id = os.path.splitext(os.path.basename(file_path))[0]
        else:
            recording_id = str(fallback_name)

    part_from_rec, block_from_rec, recording_id = parse_recording_id(recording_id)

    participant = subject if subject not in ["", "unknown", "nan", "None"] else part_from_rec
    if participant in ["", "unknown", "nan", "None"]:
        participant = str(fallback_name)

    if block is None:
        block = block_from_rec

    return participant, block, recording_id, file_path

In [3]:
# Load all saved segment metadata files and combine them into one table.

meta_paths = sorted(glob.glob(META_GLOB))
if len(meta_paths) == 0:
    raise RuntimeError(
        f"No meta PKLs found at: {META_GLOB}\n"
        "Run the main pipeline notebook first (BLOCK 0)."
    )

# If recordings_index.CSV is available, keep only the files that belong.
# This keeps the report aligned with the latest preprocessing run.
if os.path.exists(RECORDINGS_INDEX_PATH):
    try:
        rec_index_df = pd.read_csv(RECORDINGS_INDEX_PATH)
        if "recording_id" in rec_index_df.columns:
            expected_recordings = set(rec_index_df["recording_id"].astype(str).tolist())
            meta_paths = [
                p for p in meta_paths
                if os.path.basename(p).replace("_segments_meta.pkl", "") in expected_recordings
            ]
            print("Meta files after filtering with recordings_index.csv:", len(meta_paths))
    except Exception as e:
        print("[WARN] Could not use recordings_index.csv for filtering:", e)

if len(meta_paths) == 0:
    raise RuntimeError("No matching meta PKLs remained after filtering.")

rows = []
summary_rows = []

for pkl_path in meta_paths:
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    meta_list = obj.get("meta", [])
    fallback_recording = os.path.basename(pkl_path).replace("_segments_meta.pkl", "")

    first_meta = meta_list[0] if len(meta_list) > 0 else {"recording_id": fallback_recording}
    participant, block, recording_id, file_path = infer_identifiers_from_meta(
        first_meta,
        fallback_name=fallback_recording
    )

    summ = obj.get("summary", {})
    srow = {
        "participant": participant,
        "subject": participant,
        "block": block,
        "recording_id": recording_id,
        "file_path": file_path,
        "n_segments": int(len(meta_list)),
        "stim_tol_s": float(obj.get("stim_tol_s", np.nan)) if obj.get("stim_tol_s", None) is not None else np.nan,
        "dc_tail_s": float(obj.get("dc_tail_s", np.nan)) if obj.get("dc_tail_s", None) is not None else np.nan,
        "dc_min_s": float(obj.get("dc_min_s", np.nan)) if obj.get("dc_min_s", None) is not None else np.nan,
        "apply_dc_tail_rule": bool(obj.get("apply_dc_tail_rule", False)),
    }
    for k, v in summ.items():
        srow[f"removed_{k}"] = int(v)
    summary_rows.append(srow)

    for m in meta_list:
        participant_i, block_i, recording_id_i, file_path_i = infer_identifiers_from_meta(
            m,
            fallback_name=fallback_recording
        )

        raw_desc = str(m.get("raw_desc", "") or "")
        reason = str(m.get("reason", "") or "")

        rows.append({
            "participant": participant_i,
            "subject": participant_i,
            "block": block_i,
            "recording_id": recording_id_i,
            "file_path": file_path_i,
            "order": int(m.get("order", -1)),
            "fine_label": str(m.get("fine_label", "")),
            "raw_desc": raw_desc,
            "prev_raw_desc": str(m.get("prev_raw_desc", "") or ""),
            "next_raw_desc": str(m.get("next_raw_desc", "") or ""),
            "start_time_s": float(m.get("start_time_s", np.nan)),
            "end_time_s": float(m.get("end_time_s", np.nan)),
            "duration_s": float(m.get("duration_s", np.nan)),
            "bad": bool(m.get("bad", False)),
            "reason": reason,
            "is_task_candidate": bool(is_task_stimulus_desc(raw_desc)),
        })

segments_df = pd.DataFrame(rows)
if len(segments_df) == 0:
    raise RuntimeError("segments_df is empty. The meta files were found, but no segment rows were loaded.")

segments_df = segments_df.sort_values(
    ["participant", "block", "recording_id", "order"],
    na_position="last"
).reset_index(drop=True)

file_summary_df = pd.DataFrame(summary_rows)
if len(file_summary_df) > 0:
    removal_cols = [c for c in file_summary_df.columns if c.startswith("removed_")]
    for c in removal_cols:
        file_summary_df[c] = file_summary_df[c].fillna(0).astype(int)

    file_summary_df = file_summary_df.sort_values(
        ["participant", "block", "recording_id"],
        na_position="last"
    ).reset_index(drop=True)

print("Segments rows:", len(segments_df))
print("Participants:", segments_df["participant"].nunique())
print("Recordings:", segments_df["recording_id"].nunique())
print("Blocks:", sorted([x for x in segments_df["block"].dropna().unique().tolist()]))
segments_df.head()

Meta files after filtering with recordings_index.csv: 59
Segments rows: 7390
Participants: 31
Recordings: 59
Blocks: [1, 2]


,participant,subject,block,recording_id,file_path,order,fine_label,raw_desc,prev_raw_desc,next_raw_desc,start_time_s,end_time_s,duration_s,bad,reason,is_task_candidate
0,01_ln,01_ln,1,01_ln_block1,C:\Users\Asus\Documents\Professor Francesca St...,0,new_segment,New Segment/,,DC Correction/,0.0000,3.6600,3.6600,False,,False
1,01_ln,01_ln,1,01_ln_block1,C:\Users\Asus\Documents\Professor Francesca St...,1,dc_correction,DC Correction/,New Segment/,Stimulus/S 13,3.6600,20.9050,17.2450,False,,False
2,01_ln,01_ln,1,01_ln_block1,C:\Users\Asus\Documents\Professor Francesca St...,2,stim_no,Stimulus/S 13,DC Correction/,Stimulus/S134,20.9050,25.5098,4.6048,False,,True
3,01_ln,01_ln,1,01_ln_block1,C:\Users\Asus\Documents\Professor Francesca St...,3,tms_after_no,Stimulus/S134,Stimulus/S 13,Stimulus/S 11,25.5098,34.9780,9.4682,True,tms_segment,False
4,01_ln,01_ln,1,01_ln_block1,C:\Users\Asus\Documents\Professor Francesca St...,4,stim_left_real,Stimulus/S 11,Stimulus/S134,Stimulus/S114,34.9780,39.5732,4.5952,False,,True


In [4]:
# This cell collects the main totals and summarizes which cleaning rules removed task epochs.

cand = segments_df["is_task_candidate"].astype(bool).values
bad = segments_df["bad"].astype(bool).values

total_segments = int(len(segments_df))
total_candidate = int(cand.sum())
total_candidate_kept = int((cand & (~bad)).sum())
total_candidate_removed = int(total_candidate - total_candidate_kept)

print("=== Totals ===")
print("Total segments (all types):", total_segments)
print("Candidate TASK stimuli (before cleaning):", total_candidate)
print("Kept TASK stimuli (after cleaning):", total_candidate_kept)
print("Removed TASK stimuli:", total_candidate_removed)

def normalize_reason(r: str):
    r = str(r or "").strip()
    if r.startswith("dc_within_tail"):
        return "dc_within_tail"
    return r

# Count removals by rule, restricting the summary to task candidates only.
rem_counts = Counter()
for r in segments_df.loc[cand & bad, "reason"].fillna("").astype(str).tolist():
    if not r.strip():
        rem_counts["(unspecified)"] += 1
        continue

    parts = [normalize_reason(x.strip()) for x in r.split("|") if x.strip()]
    if len(parts) == 0:
        rem_counts["(unspecified)"] += 1
    else:
        for x in parts:
            rem_counts[x] += 1

rem_df = pd.DataFrame(
    rem_counts.items(),
    columns=["rule", "removed_count"]
).sort_values("removed_count", ascending=False).reset_index(drop=True)

print("\n=== Removed TASK epochs by rule (counts) ===")
if len(rem_df) == 0:
    print("No removed TASK epochs found.")
else:
    print(rem_df.head(30).to_string(index=False))

# Build a recording level summary for task candidates.
cand_df = segments_df.loc[cand].copy()

per_recording_df = cand_df.groupby(
    ["participant", "block", "recording_id"],
    dropna=False
).agg(
    candidate_task=("order", "count"),
    kept_task=("bad", lambda x: int((~x.astype(bool)).sum())),
).reset_index()

per_recording_df["removed_task"] = per_recording_df["candidate_task"] - per_recording_df["kept_task"]
per_recording_df["removed_fraction"] = np.where(
    per_recording_df["candidate_task"] > 0,
    per_recording_df["removed_task"] / per_recording_df["candidate_task"],
    np.nan
)

per_recording_df = per_recording_df.sort_values(
    ["participant", "block", "recording_id"],
    na_position="last"
).reset_index(drop=True)

# Save the main reporting tables.
out1 = os.path.join(OUT_DIR, "C_segments_table_all.csv")
out2 = os.path.join(OUT_DIR, "C_task_removals_by_rule.csv")
out3 = os.path.join(OUT_DIR, "C_task_summary_per_recording.csv")
out4 = os.path.join(OUT_DIR, "C_file_summary_from_state_machine.csv")

segments_df.to_csv(out1, index=False)
rem_df.to_csv(out2, index=False)
per_recording_df.to_csv(out3, index=False)
file_summary_df.to_csv(out4, index=False)

print("\nSaved:")
print(" -", out1)
print(" -", out2)
print(" -", out3)
print(" -", out4)

per_recording_df.head(10)

=== Totals ===
Total segments (all types): 7390
Candidate TASK stimuli (before cleaning): 1740
Kept TASK stimuli (after cleaning): 1711
Removed TASK stimuli: 29

=== Removed TASK epochs by rule (counts) ===
                  rule  removed_count
stim_interrupted_by_dc             16
        dc_within_tail             13

Saved:
 - D:\EEG_CleanSegments_And_Datasets_AUX\C_segments_table_all.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\C_task_removals_by_rule.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\C_task_summary_per_recording.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\C_file_summary_from_state_machine.csv


,participant,block,recording_id,candidate_task,kept_task,removed_task,removed_fraction
0,01_ln,1,01_ln_block1,60,60,0,0.000000
1,02gd,1,02gd_block1,60,59,1,0.016667
2,03as,1,03as_block1,60,55,5,0.083333
3,04ep,1,04ep_block1,60,60,0,0.000000
4,06nc,1,06nc_block1,60,60,0,0.000000
5,07fs,1,07fs_block1,60,60,0,0.000000
6,08am,1,08am_block1,60,60,0,0.000000
7,09dm,1,09dm_block1,60,59,1,0.016667
8,10lc,1,10lc_block1,60,59,1,0.016667
9,11gb,1,11gb_block1,60,60,0,0.000000


In [5]:
# This cell estimates a reasonable STIM_TOL_S directly from the observed stimulus durations.
# And saves both the summary tables and the figure outputs used for reporting.

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

STIM_REPORT_DIR = os.path.join(OUT_DIR, "_stim_tol_report")
os.makedirs(STIM_REPORT_DIR, exist_ok=True)

EXPECTED_STIM_S = 4.5
PLAUSIBLE_MIN_S = 3.5
PLAUSIBLE_MAX_S = 5.5
ROUND_STEP_S = 0.05

task_df = segments_df.loc[segments_df["is_task_candidate"] == True].copy()
task_df = task_df[np.isfinite(task_df["duration_s"].values)].copy()

if len(task_df) == 0:
    raise RuntimeError("No task-candidate rows found in segments_df.")

# A broad window around 4.5 s is used here so clearly interrupted epochs do not drive the estimate.
plausible_df = task_df.loc[
    (task_df["duration_s"] >= PLAUSIBLE_MIN_S) &
    (task_df["duration_s"] <= PLAUSIBLE_MAX_S)
].copy()

if len(plausible_df) == 0:
    raise RuntimeError(
        "No plausible task durations found in the broad window "
        f"[{PLAUSIBLE_MIN_S}, {PLAUSIBLE_MAX_S}] s."
    )

plausible_df["abs_dev_s"] = np.abs(plausible_df["duration_s"] - EXPECTED_STIM_S)

q = plausible_df["abs_dev_s"].quantile([0.50, 0.90, 0.95, 0.99]).to_dict()
p50 = float(q.get(0.50, np.nan))
p90 = float(q.get(0.90, np.nan))
p95 = float(q.get(0.95, np.nan))
p99 = float(q.get(0.99, np.nan))

recommended_tol = float(np.ceil(p99 / ROUND_STEP_S) * ROUND_STEP_S)
recommended_tol = max(0.05, recommended_tol)

# Keep counts are shown for the old tolerance and the newly estimated one.
legacy_tol = 0.50
legacy_keep = int(((task_df["duration_s"] >= EXPECTED_STIM_S - legacy_tol) &
                   (task_df["duration_s"] <= EXPECTED_STIM_S + legacy_tol)).sum())
new_keep = int(((task_df["duration_s"] >= EXPECTED_STIM_S - recommended_tol) &
                (task_df["duration_s"] <= EXPECTED_STIM_S + recommended_tol)).sum())

summary_df = pd.DataFrame([
    {"metric": "n_task_candidates_total", "value": int(len(task_df))},
    {"metric": "n_plausible_used_for_estimation", "value": int(len(plausible_df))},
    {"metric": "expected_stim_s", "value": float(EXPECTED_STIM_S)},
    {"metric": "plausible_min_s", "value": float(PLAUSIBLE_MIN_S)},
    {"metric": "plausible_max_s", "value": float(PLAUSIBLE_MAX_S)},
    {"metric": "abs_dev_p50_s", "value": p50},
    {"metric": "abs_dev_p90_s", "value": p90},
    {"metric": "abs_dev_p95_s", "value": p95},
    {"metric": "abs_dev_p99_s", "value": p99},
    {"metric": "recommended_STIM_TOL_S", "value": float(recommended_tol)},
    {"metric": "legacy_STIM_TOL_S", "value": float(legacy_tol)},
    {"metric": "legacy_keep_count", "value": int(legacy_keep)},
    {"metric": "recommended_keep_count", "value": int(new_keep)},
])

summary_csv = os.path.join(STIM_REPORT_DIR, "stim_tol_summary.csv")
summary_df.to_csv(summary_csv, index=False)

# These examples are useful for inspecting durations that sit near the edge of the plausible range.
borderline_df = plausible_df.sort_values("abs_dev_s", ascending=False).head(40).copy()
borderline_csv = os.path.join(STIM_REPORT_DIR, "stim_tol_borderline_examples.csv")
borderline_df.to_csv(borderline_csv, index=False)

# Figure 1: distribution of task candidate durations with the recommended acceptance window.
plt.figure(figsize=(7, 4))
plt.hist(task_df["duration_s"].values, bins=50)
plt.axvline(EXPECTED_STIM_S, linestyle="--", label="Expected 4.5 s")
plt.axvline(EXPECTED_STIM_S - recommended_tol, linestyle=":", label=f"Recommended lower ({EXPECTED_STIM_S - recommended_tol:.2f})")
plt.axvline(EXPECTED_STIM_S + recommended_tol, linestyle=":", label=f"Recommended upper ({EXPECTED_STIM_S + recommended_tol:.2f})")
plt.xlabel("Task-candidate duration (s)")
plt.ylabel("Count")
plt.title("Task-candidate duration distribution")
plt.legend()
plt.tight_layout()
dur_hist_png = os.path.join(STIM_REPORT_DIR, "stim_tol_duration_histogram.png")
plt.savefig(dur_hist_png, dpi=180)
plt.close()

# Figure 2: distribution of absolute deviations from the nominal 4.5 s duration.
plt.figure(figsize=(7, 4))
plt.hist(plausible_df["abs_dev_s"].values, bins=40)
plt.axvline(p95, linestyle="--", label=f"p95={p95:.3f}s")
plt.axvline(p99, linestyle="--", label=f"p99={p99:.3f}s")
plt.axvline(recommended_tol, linestyle=":", label=f"recommended={recommended_tol:.3f}s")
plt.xlabel("|duration - 4.5s| (s)")
plt.ylabel("Count")
plt.title("Absolute deviation from expected stimulus length")
plt.legend()
plt.tight_layout()
absdev_png = os.path.join(STIM_REPORT_DIR, "stim_tol_abs_deviation_histogram.png")
plt.savefig(absdev_png, dpi=180)
plt.close()

# Save the recommended tolerance so the preprocessing notebook can load it directly.
stim_cfg = {
    "schema_version": 1,
    "source_notebook": "03_dataset_report_ablation_2blocks.ipynb",
    "expected_stim_s": float(EXPECTED_STIM_S),
    "plausible_min_s": float(PLAUSIBLE_MIN_S),
    "plausible_max_s": float(PLAUSIBLE_MAX_S),
    "abs_dev_p50_s": p50,
    "abs_dev_p90_s": p90,
    "abs_dev_p95_s": p95,
    "abs_dev_p99_s": p99,
    "round_step_s": float(ROUND_STEP_S),
    "STIM_TOL_S": float(recommended_tol),
    "n_task_candidates_total": int(len(task_df)),
    "n_plausible_used_for_estimation": int(len(plausible_df)),
    "note": "Tolerance estimated from plausible task-candidate durations near the nominal 4.5 s stimulus length."
}
stim_json = os.path.join(OUT_DIR, "STIM_TOL_S_recommended.json")
with open(stim_json, "w", encoding="utf-8") as f:
    json.dump(stim_cfg, f, indent=2)

print("=== STIM_TOL_S ESTIMATION ===")
print("Task candidates total:", len(task_df))
print("Plausible durations used:", len(plausible_df))
print("abs_dev p95:", round(p95, 4), "s")
print("abs_dev p99:", round(p99, 4), "s")
print("Recommended STIM_TOL_S:", round(recommended_tol, 4), "s")
print("\nSaved:")
print(" -", summary_csv)
print(" -", borderline_csv)
print(" -", dur_hist_png)
print(" -", absdev_png)
print(" -", stim_json)

=== STIM_TOL_S ESTIMATION ===
Task candidates total: 1740
Plausible durations used: 1725
abs_dev p95: 0.242 s
abs_dev p99: 0.2543 s
Recommended STIM_TOL_S: 0.3 s

Saved:
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stim_tol_report\stim_tol_summary.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stim_tol_report\stim_tol_borderline_examples.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stim_tol_report\stim_tol_duration_histogram.png
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stim_tol_report\stim_tol_abs_deviation_histogram.png
 - D:\EEG_CleanSegments_And_Datasets_AUX\STIM_TOL_S_recommended.json


In [6]:
# These plots give a quick visual summary of what was removed and how many task epochs.
# Remained in each recording after cleaning.

cand = segments_df["is_task_candidate"].astype(bool).values
bad = segments_df["bad"].astype(bool).values

removed_hist_png = os.path.join(PLOT_DIR, "removed_task_candidate_duration_histogram.png")
kept_bar_png = os.path.join(PLOT_DIR, "kept_task_epochs_per_recording.png")

plt.figure(figsize=(7, 4))
plt.hist(segments_df.loc[cand & bad, "duration_s"].dropna().values, bins=40)
plt.xlabel("Duration (s) of removed TASK candidate segments")
plt.ylabel("Count")
plt.title("Removed TASK candidates: duration distribution")
plt.tight_layout()
plt.savefig(removed_hist_png, dpi=180)
plt.close()

cand_df = segments_df.loc[cand].copy()
group_col = "recording_id" if "recording_id" in cand_df.columns else "subject"
per_keep = cand_df.groupby(group_col)["bad"].apply(lambda x: int((~x.astype(bool)).sum())).sort_index()

plt.figure(figsize=(max(10, 0.4 * len(per_keep)), 4))
plt.bar(np.arange(len(per_keep)), per_keep.values)
plt.xticks(np.arange(len(per_keep)), per_keep.index, rotation=90, fontsize=7)
plt.ylabel("Kept TASK epochs")
plt.title("Kept TASK epochs per recording")
plt.tight_layout()
plt.savefig(kept_bar_png, dpi=180)
plt.close()

print("Saved:")
print(" -", removed_hist_png)
print(" -", kept_bar_png)

Saved:
 - D:\EEG_CleanSegments_And_Datasets_AUX\_dataset_report\plots\removed_task_candidate_duration_histogram.png
 - D:\EEG_CleanSegments_And_Datasets_AUX\_dataset_report\plots\kept_task_epochs_per_recording.png


In [7]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal

STFT_REPORT_DIR = os.path.join(OUT_DIR, "_stft_report")
os.makedirs(STFT_REPORT_DIR, exist_ok=True)

TIME_NPZ = os.path.join(OUT_DIR, "clean_time_dataset.npz")
if not os.path.exists(TIME_NPZ):
    raise FileNotFoundError(f"Missing dataset: {TIME_NPZ}")

d = np.load(TIME_NPZ, allow_pickle=True)
X = d["X"].astype(np.float32)   # Shape: (N, C, T).
y = d["y"].astype(np.int64)
sfreq = float(d["sfreq"][0]) if "sfreq" in d.files else 250.0

# Default STFT values used in case no saved configuration is available yet.
CHOSEN_STFT_WINDOW = "hann"
CHOSEN_STFT_NPERSEG_S = 3.0
CHOSEN_STFT_OVERLAP_S = 2.8
CHOSEN_STFT_NFFT_MULT = 4
CHOSEN_STFT_MAX_FREQ_HZ = 40.0

# If the main preprocessing run already saved the final STFT settings,.
# Load them here so the report matches the actual pipeline exactly.
if os.path.exists(BUILD_SUMMARY_PATH):
    try:
        with open(BUILD_SUMMARY_PATH, "r", encoding="utf-8") as f:
            build_cfg = json.load(f)

        CHOSEN_STFT_WINDOW = str(build_cfg.get("stft_window", CHOSEN_STFT_WINDOW))
        CHOSEN_STFT_NPERSEG_S = float(build_cfg.get("stft_nperseg_s", CHOSEN_STFT_NPERSEG_S))
        CHOSEN_STFT_OVERLAP_S = float(build_cfg.get("stft_overlap_s", CHOSEN_STFT_OVERLAP_S))
        CHOSEN_STFT_NFFT_MULT = int(build_cfg.get("stft_nfft_mult", CHOSEN_STFT_NFFT_MULT))
        CHOSEN_STFT_MAX_FREQ_HZ = float(build_cfg.get("stft_max_freq_hz", CHOSEN_STFT_MAX_FREQ_HZ))
        print("[STFT] Loaded chosen settings from dataset_build_summary.json")
    except Exception as e:
        print("[STFT] Could not read dataset_build_summary.json, using defaults:", e)

NPERSEG_CANDIDATES_S = [1.0, 2.0, 3.0]
OVERLAP_CANDIDATES_S = [0.5, 1.0, 2.0, 2.8]
MAX_PLOT_FREQ_HZ = 60.0

# Take one clean example from each class for visual comparison.
example_indices = []
for cls in sorted(np.unique(y).tolist()):
    idx = np.where(y == cls)[0]
    if len(idx) > 0:
        example_indices.append(int(idx[0]))

if len(example_indices) == 0:
    raise RuntimeError("No examples found in clean_time_dataset.npz.")

channel_idx = int(X.shape[1] // 2)

def _compute_stft(x_1d, sf, nperseg_s, overlap_s, window="hann", nfft_mult=4):
    nperseg = int(round(nperseg_s * sf))
    noverlap = int(round(overlap_s * sf))
    noverlap = min(noverlap, nperseg - 1)
    nfft = int(nfft_mult * nperseg)

    f, t, Z = signal.stft(
        x_1d,
        fs=sf,
        window=window,
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        boundary=None,
        padded=False,
    )
    return f, t, np.abs(Z)

# Save the final parameter choice in a simple table for reference.
choice_summary_df = pd.DataFrame([{
    "STFT_WINDOW": CHOSEN_STFT_WINDOW,
    "STFT_NPERSEG_S": float(CHOSEN_STFT_NPERSEG_S),
    "STFT_OVERLAP_S": float(CHOSEN_STFT_OVERLAP_S),
    "STFT_NFFT_MULT": int(CHOSEN_STFT_NFFT_MULT),
    "STFT_MAX_FREQ_HZ": float(CHOSEN_STFT_MAX_FREQ_HZ),
    "sfreq": float(sfreq),
}])
choice_summary_csv = os.path.join(STFT_REPORT_DIR, "stft_chosen_settings_summary.csv")
choice_summary_df.to_csv(choice_summary_csv, index=False)

# Compare different window lengths while keeping the chosen settings otherwise close.
for ex_i in example_indices:
    cls = int(y[ex_i])
    x = X[ex_i, channel_idx]

    fig, axes = plt.subplots(1, len(NPERSEG_CANDIDATES_S), figsize=(5 * len(NPERSEG_CANDIDATES_S), 4), sharey=True)
    if len(NPERSEG_CANDIDATES_S) == 1:
        axes = [axes]

    for ax, nps in zip(axes, NPERSEG_CANDIDATES_S):
        ov = min(CHOSEN_STFT_OVERLAP_S, nps - 0.1)
        f, t, mag = _compute_stft(
            x, sfreq, nps, ov,
            window=CHOSEN_STFT_WINDOW,
            nfft_mult=CHOSEN_STFT_NFFT_MULT
        )
        keep = f <= MAX_PLOT_FREQ_HZ
        ax.pcolormesh(t, f[keep], mag[keep], shading="auto")
        ax.set_title(f"class={cls} | nperseg={nps:.1f}s\noverlap={ov:.1f}s")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Freq (Hz)")

    plt.tight_layout()
    out_png = os.path.join(STFT_REPORT_DIR, f"stft_compare_nperseg_class{cls}.png")
    plt.savefig(out_png, dpi=180)
    plt.close()

# Compare different overlap values while keeping the chosen window length fixed.
for ex_i in example_indices:
    cls = int(y[ex_i])
    x = X[ex_i, channel_idx]

    fig, axes = plt.subplots(1, len(OVERLAP_CANDIDATES_S), figsize=(5 * len(OVERLAP_CANDIDATES_S), 4), sharey=True)
    if len(OVERLAP_CANDIDATES_S) == 1:
        axes = [axes]

    for ax, ovs in zip(axes, OVERLAP_CANDIDATES_S):
        nps = CHOSEN_STFT_NPERSEG_S
        ov = min(ovs, nps - 0.1)
        f, t, mag = _compute_stft(
            x, sfreq, nps, ov,
            window=CHOSEN_STFT_WINDOW,
            nfft_mult=CHOSEN_STFT_NFFT_MULT
        )
        keep = f <= MAX_PLOT_FREQ_HZ
        ax.pcolormesh(t, f[keep], mag[keep], shading="auto")
        ax.set_title(f"class={cls} | overlap={ov:.1f}s\nnperseg={nps:.1f}s")
        ax.set_xlabel("Time (s)")
        ax.set_ylabel("Freq (Hz)")

    plt.tight_layout()
    out_png = os.path.join(STFT_REPORT_DIR, f"stft_compare_overlap_class{cls}.png")
    plt.savefig(out_png, dpi=180)
    plt.close()

# Estimate the average PSD across a subset of clean epochs to support the frequency cutoff choice.
MAX_EPOCHS_FOR_PSD = min(250, X.shape[0])
subset_idx = np.arange(MAX_EPOCHS_FOR_PSD)

psd_list = []
freq_ref = None

for i in subset_idx:
    xi = X[i]
    for c in range(xi.shape[0]):
        f_psd, pxx = signal.welch(
            xi[c],
            fs=sfreq,
            window="hann",
            nperseg=min(512, xi.shape[1]),
            noverlap=min(256, max(0, xi.shape[1] // 2)),
            scaling="density"
        )
        if freq_ref is None:
            freq_ref = f_psd
        psd_list.append(pxx)

psd_arr = np.stack(psd_list, axis=0)
mean_psd = np.mean(psd_arr, axis=0)

psd_df = pd.DataFrame({
    "freq_hz": freq_ref,
    "mean_psd": mean_psd,
})
psd_csv = os.path.join(STFT_REPORT_DIR, "average_psd_clean_epochs.csv")
psd_df.to_csv(psd_csv, index=False)

plt.figure(figsize=(7, 4))
plt.plot(freq_ref, mean_psd)
plt.axvline(CHOSEN_STFT_MAX_FREQ_HZ, linestyle="--", label=f"{CHOSEN_STFT_MAX_FREQ_HZ:.0f} Hz")
plt.xlim(0, 100)
plt.xlabel("Frequency (Hz)")
plt.ylabel("Mean PSD")
plt.title("Average PSD over clean epochs/channels")
plt.legend()
plt.tight_layout()
psd_png = os.path.join(STFT_REPORT_DIR, "average_psd_clean_epochs.png")
plt.savefig(psd_png, dpi=180)
plt.close()

thresholds = [20, 30, 40, 60, 80, 100]
total_energy = float(np.trapz(mean_psd, freq_ref))
energy_rows = []

for thr in thresholds:
    keep = freq_ref <= thr
    e_thr = float(np.trapz(mean_psd[keep], freq_ref[keep]))
    frac = (e_thr / total_energy) if total_energy > 0 else np.nan
    energy_rows.append({
        "max_freq_hz": float(thr),
        "energy_fraction_leq_thr": float(frac) if np.isfinite(frac) else np.nan,
    })

energy_df = pd.DataFrame(energy_rows)
energy_csv = os.path.join(STFT_REPORT_DIR, "stft_max_freq_energy_fraction.csv")
energy_df.to_csv(energy_csv, index=False)

# Export the selected STFT settings so the main preprocessing notebook can reuse them directly.
stft_cfg = {
    "schema_version": 1,
    "source_notebook": "03_dataset_report_ablation.ipynb",
    "STFT_WINDOW": CHOSEN_STFT_WINDOW,
    "STFT_NPERSEG_S": float(CHOSEN_STFT_NPERSEG_S),
    "STFT_OVERLAP_S": float(CHOSEN_STFT_OVERLAP_S),
    "STFT_NFFT_MULT": int(CHOSEN_STFT_NFFT_MULT),
    "STFT_MAX_FREQ_HZ": float(CHOSEN_STFT_MAX_FREQ_HZ),
    "comparison_nperseg_candidates_s": [float(x) for x in NPERSEG_CANDIDATES_S],
    "comparison_overlap_candidates_s": [float(x) for x in OVERLAP_CANDIDATES_S],
    "psd_energy_fraction_leq_40hz": float(
        energy_df.loc[energy_df["max_freq_hz"] == 40, "energy_fraction_leq_thr"].iloc[0]
    ),
    "note": "Chosen STFT settings were retained after visual comparison of candidate spectrogram parameterizations and average PSD inspection on clean epochs."
}

stft_json = os.path.join(OUT_DIR, "STFT_recommended.json")
with open(stft_json, "w", encoding="utf-8") as f:
    json.dump(stft_cfg, f, indent=2)

stft_json_copy = os.path.join(STFT_REPORT_DIR, "STFT_recommended.json")
with open(stft_json_copy, "w", encoding="utf-8") as f:
    json.dump(stft_cfg, f, indent=2)

print("=== STFT JUSTIFICATION REPORT ===")
print("Saved:")
print(" -", choice_summary_csv)
print(" -", psd_csv)
print(" -", psd_png)
print(" -", energy_csv)
print(" -", stft_json)
print(" -", stft_json_copy)
print(" - multiple comparison PNGs in:", STFT_REPORT_DIR)

[STFT] Loaded chosen settings from dataset_build_summary.json
=== STFT JUSTIFICATION REPORT ===
Saved:
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report\stft_chosen_settings_summary.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report\average_psd_clean_epochs.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report\average_psd_clean_epochs.png
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report\stft_max_freq_energy_fraction.csv
 - D:\EEG_CleanSegments_And_Datasets_AUX\STFT_recommended.json
 - D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report\STFT_recommended.json
 - multiple comparison PNGs in: D:\EEG_CleanSegments_And_Datasets_AUX\_stft_report


In [8]:
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score

def load_npz_dataset(path_npz):
    d = np.load(path_npz, allow_pickle=True)
    X = d["X"]
    y = d["y"].astype(np.int64)
    groups = d["subject"].astype(object)
    return X, y, groups

# Choose whether to report the original 3 class task or only left vs right.
MODE = "3class"

def remap_labels(y):
    y = np.asarray(y, dtype=np.int64)

    if MODE == "3class":
        keep = np.isin(y, [0, 1, 2])
        y_out = y.copy()
        return keep, y_out

    elif MODE == "binary_lr":
        keep = np.isin(y, [0, 1])
        y_out = y.copy()
        return keep, y_out

    else:
        raise ValueError(f"Unknown MODE: {MODE}")

def eval_logreg_grouped_cv(X, y, groups, seed=42, max_n_splits=5, max_pca_components=100):
    """
    Run a simple grouped cross-validation baseline.

    The split is grouped by subject, PCA is fit inside each training fold,
    and folds that do not contain enough class variation are skipped.
    """
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    groups = np.asarray(groups, dtype=object)

    if X.ndim != 3:
        raise ValueError(f"Expected X with shape (N,C,T), got {X.shape}")

    Xf = X.reshape(X.shape[0], -1)

    uniq_groups = np.unique(groups)
    n_splits = min(max_n_splits, len(uniq_groups))
    if n_splits < 2:
        raise RuntimeError(f"Need at least 2 unique groups for CV, got {len(uniq_groups)}")

    gkf = GroupKFold(n_splits=n_splits)

    accs = []
    f1s = []
    fold_rows = []

    for fold_idx, (tr, te) in enumerate(gkf.split(Xf, y, groups), start=1):
        X_tr, X_te = Xf[tr], Xf[te]
        y_tr, y_te = y[tr], y[te]

        tr_classes = np.unique(y_tr)
        te_classes = np.unique(y_te)

        if tr_classes.size < 2:
            print(f"[SKIP] Fold {fold_idx}: training split has <2 classes")
            continue
        if te_classes.size < 2:
            print(f"[SKIP] Fold {fold_idx}: test split has <2 classes")
            continue

        # PCA dimensionality is chosen from the current training fold only.
        n_comp = min(
            max_pca_components,
            X_tr.shape[0] - 1,
            X_tr.shape[1]
        )
        n_comp = max(2, int(n_comp))

        pipe = Pipeline([
            ("scaler", StandardScaler(with_mean=True, with_std=True)),
            ("pca", PCA(n_components=n_comp, random_state=seed)),
            ("clf", LogisticRegression(
                solver="saga",
                max_iter=3000,
                random_state=seed,
                n_jobs=-1
            ))
        ])

        pipe.fit(X_tr, y_tr)
        yp = pipe.predict(X_te)

        fold_acc = accuracy_score(y_te, yp)
        fold_f1 = f1_score(y_te, yp, average="macro")

        accs.append(float(fold_acc))
        f1s.append(float(fold_f1))
        fold_rows.append({
            "fold": int(fold_idx),
            "n_train": int(len(tr)),
            "n_test": int(len(te)),
            "n_groups_train": int(len(np.unique(groups[tr]))),
            "n_groups_test": int(len(np.unique(groups[te]))),
            "n_pca_components": int(n_comp),
            "acc": float(fold_acc),
            "macro_f1": float(fold_f1),
        })

    if len(accs) == 0:
        raise RuntimeError("No valid CV folds were available for ablation evaluation.")

    fold_df = pd.DataFrame(fold_rows)
    return {
        "mean_acc": float(np.mean(accs)),
        "std_acc": float(np.std(accs)),
        "mean_macro_f1": float(np.mean(f1s)),
        "std_macro_f1": float(np.std(f1s)),
        "n_valid_folds": int(len(accs)),
        "fold_df": fold_df,
    }

if (not os.path.exists(NPZ_NO_DCTAIL)) or (not os.path.exists(NPZ_WITH_DCTAIL)):
    print("Ablation NPZ files not found.")
    print("Expected:")
    print(" -", NPZ_NO_DCTAIL)
    print(" -", NPZ_WITH_DCTAIL)
    print("\nTo create them: run the main pipeline twice and save outputs with these names.")
else:
    X0, y0, g0 = load_npz_dataset(NPZ_NO_DCTAIL)
    X1, y1, g1 = load_npz_dataset(NPZ_WITH_DCTAIL)

    k0, y0m = remap_labels(y0)
    k1, y1m = remap_labels(y1)

    res0 = eval_logreg_grouped_cv(X0[k0], y0m[k0], g0[k0], seed=42)
    res1 = eval_logreg_grouped_cv(X1[k1], y1m[k1], g1[k1], seed=42)

    print("=== Ablation (LogReg + PCA, grouped CV) ===")
    print("MODE:", MODE)
    print()
    print("NO DC-tail")
    print("  mean_acc      =", round(res0["mean_acc"], 4))
    print("  std_acc       =", round(res0["std_acc"], 4))
    print("  mean_macroF1  =", round(res0["mean_macro_f1"], 4))
    print("  std_macroF1   =", round(res0["std_macro_f1"], 4))
    print("  valid_folds   =", res0["n_valid_folds"])
    print()
    print("WITH DC-tail")
    print("  mean_acc      =", round(res1["mean_acc"], 4))
    print("  std_acc       =", round(res1["std_acc"], 4))
    print("  mean_macroF1  =", round(res1["mean_macro_f1"], 4))
    print("  std_macroF1   =", round(res1["std_macro_f1"], 4))
    print("  valid_folds   =", res1["n_valid_folds"])

    out_fold_no = os.path.join(OUT_DIR, f"C_ablation_logreg_pca_{MODE}_no_dctail_folds.csv")
    out_fold_yes = os.path.join(OUT_DIR, f"C_ablation_logreg_pca_{MODE}_with_dctail_folds.csv")
    out_summary = os.path.join(OUT_DIR, f"C_ablation_logreg_pca_{MODE}_summary.csv")

    res0["fold_df"].to_csv(out_fold_no, index=False)
    res1["fold_df"].to_csv(out_fold_yes, index=False)

    summary_df = pd.DataFrame([
        {
            "dataset": "no_dctail",
            "mode": MODE,
            "mean_acc": res0["mean_acc"],
            "std_acc": res0["std_acc"],
            "mean_macro_f1": res0["mean_macro_f1"],
            "std_macro_f1": res0["std_macro_f1"],
            "n_valid_folds": res0["n_valid_folds"],
        },
        {
            "dataset": "with_dctail",
            "mode": MODE,
            "mean_acc": res1["mean_acc"],
            "std_acc": res1["std_acc"],
            "mean_macro_f1": res1["mean_macro_f1"],
            "std_macro_f1": res1["std_macro_f1"],
            "n_valid_folds": res1["n_valid_folds"],
        },
    ])
    summary_df.to_csv(out_summary, index=False)

    print("\nSaved:")
    print(" -", out_fold_no)
    print(" -", out_fold_yes)
    print(" -", out_summary)

Ablation NPZ files not found.
Expected:
 - D:\EEG_CleanSegments_And_Datasets_AUX\clean_time_dataset_no_dctail.npz
 - D:\EEG_CleanSegments_And_Datasets_AUX\clean_time_dataset_with_dctail.npz

To create them: run the main pipeline twice and save outputs with these names.
